# 16 — SU steering's effect on the eval-aware axis (Qwen3.5-4B)

**Question.** When we apply S/U steering at the magnitude that jailbreaks (`SU −0.7σ`, peak from exp15), does the model's *eval-aware* readout move toward the deployed-mode end of the axis, or does it stay put?

Yesterday we already showed that **SU and eval-aware vectors are near-orthogonal** (low cos). That rules out *geometric* overlap. It does **not** rule out **circuit-level coupling** — some MLP/attention head reading from SU and writing to eval-aware downstream. This notebook tests for that coupling by:

1. Forward-passing AdvBench-30 (and a held-out neutral set) under each condition with `ResidualSteerer` installed (no generation).
2. Capturing the post-block residual at every layer in `[16..31]` via `output_hidden_states=True`.
3. Projecting onto the eval-aware unit direction at each layer, plus refusal direction as a control.
4. Overlaying the `test_mean` / `deploy_mean` anchors per layer so the steered readout has a calibrated location on the test↔deploy axis.

**Why this is clean.** Vectors are ~orthogonal → direct-bleed (the SU `δ` vector mechanically projecting onto eval) is ~0. Any non-zero shift in the eval-axis readout under steering is *necessarily* mediated through downstream computation — actual circuit coupling.

**Conditions.** Baseline + `SU −0.5σ` (no jailbreak) + `SU −0.65σ` (cliff edge) + `SU −0.7σ` (peak) + `SU −0.85σ` (post-cliff, near OOD) + `refusal −0.4σ` (refusal-axis jailbreak peak) + `refusal +0.3σ` (defense booster).

**Self-contained / Colab-portable.** Same clone-and-load pattern as exp15. ~30 prompts × 7 conditions × 2 pools = ~420 forward passes (no generation). Few minutes on a single A100.

**Inputs needed.**
- `exp06_pca_directions.npz` — S/U PCA directions (steerable).
- `exp06_results/arrays.npz` — S/U mean-diff raw vectors (for σ).
- `exp_directions_qwen35_4b/directions.npz` — refusal directions (steerable) + eval-aware directions (readout only) + per-class means (anchors).
- `data/advbench_harmful.json` — AdvBench-30 prompts.

## 0 — GPU check

In [ ]:
import subprocess, torch
try:
    print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout)
except FileNotFoundError:
    print('no nvidia-smi (CPU/MPS host?)')
if torch.cuda.is_available():
    n = torch.cuda.device_count()
    total_gb = sum(torch.cuda.get_device_properties(i).total_memory for i in range(n)) / 1e9
    print(f'{n} GPUs visible, total VRAM = {total_gb:.0f} GB')
else:
    print('no CUDA; will run on CPU/MPS — slow but correct.')

## 1 — Install dependencies

In [ ]:
!pip install -q 'transformers>=4.45' 'accelerate>=0.33' huggingface_hub tqdm numpy pandas matplotlib

In [ ]:
!pip install transformers --ugrade 

## 1b — Clone the Mech_spoof repo (if not already on disk)

On a fresh Colab/pod we need the repo for source code, directions, and prompts. Skip this cell if you've rsynced the repo already and `MECH_SPOOF_ROOT` is set.

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/ChuloIva/Mech_spoof.git'

if os.environ.get('MECH_SPOOF_ROOT'):
    target = Path(os.environ['MECH_SPOOF_ROOT'])
elif Path('/content').exists():
    target = Path('/content/Mech_spoof')
elif Path('/workspace').exists():
    target = Path('/workspace/Mech_spoof')
else:
    target = Path.cwd() / 'Mech_spoof'

if not (target / 'src' / 'mech_spoof').exists():
    target.parent.mkdir(parents=True, exist_ok=True)
    print(f'cloning {REPO_URL} → {target}')
    !git clone --depth 1 {REPO_URL} {target}
else:
    print(f'repo already at {target} (skipping clone)')

os.environ['MECH_SPOOF_ROOT'] = str(target)

for sub in ['src/mech_spoof',
            'exp06_pca_directions.npz',
            'exp06_results/arrays.npz',
            'exp_directions_qwen35_4b/directions.npz',
            'data/advbench_harmful.json']:
    p = target / sub
    print(f'  {sub:<45s} {"OK" if p.exists() else "MISSING"}')

## 2 — Project root, paths, optional HF auth

In [ ]:
import os, sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path(os.environ.get('MECH_SPOOF_ROOT', '/workspace/Mech_spoof'))
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    print(f'falling back to PROJECT_ROOT={PROJECT_ROOT}')
assert (PROJECT_ROOT / 'src' / 'mech_spoof').exists(), f'no mech_spoof at {PROJECT_ROOT}/src — set MECH_SPOOF_ROOT'

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

EXP6_PCA    = PROJECT_ROOT / 'exp06_pca_directions.npz'
EXP6_ARRAYS = PROJECT_ROOT / 'exp06_results' / 'arrays.npz'
DIRS_NPZ    = PROJECT_ROOT / 'exp_directions_qwen35_4b' / 'directions.npz'
ADVBENCH    = PROJECT_ROOT / 'data' / 'advbench_harmful.json'
OUT_DIR     = PROJECT_ROOT / 'exp16_su_eval_readout'
OUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import userdata
    try:
        os.environ.setdefault('HF_TOKEN', userdata.get('HF_TOKEN'))
    except Exception:
        pass
except Exception:
    pass
if os.environ.get('HF_TOKEN'):
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)

for label, p in [('exp06_pca', EXP6_PCA), ('exp06_arrays', EXP6_ARRAYS),
                  ('dirs_npz', DIRS_NPZ), ('advbench', ADVBENCH)]:
    print(f'  {label:<14s} {p}  {"OK" if p.exists() else "MISSING"}')
print('  out_dir       ', OUT_DIR)

## 3 — Load Qwen 3.5 4B

In [ ]:
from mech_spoof.io import load_npz
from mech_spoof.models import load_model
from mech_spoof.probes import ResidualSteerer

loaded = load_model('qwen')
model, tok = loaded.hf_model, loaded.tokenizer
device = loaded.device
model.eval()
supports_thinking = getattr(loaded.template, '_supports_enable_thinking', False)
print(f'model={loaded.cfg.hf_id}  device={device}  n_layers={loaded.n_layers}  d_model={loaded.d_model}  '
      f'thinking_supported={supports_thinking}')

## 4 — Build direction registry

Three roles for the directions:

- **`SU/exp06_pca_center`** (steerable). Same as exp15, layers 16..31, position `response_last`.
- **`refusal/pos-3`** (steerable). Same as exp15.
- **`eval_aw/contrastive_dataset`** (READOUT ONLY). Per-layer unit direction + per-class means (`test_mean`, `deploy_mean`) used as anchors when interpreting projections.

We also compute `cos(SU, eval_aw)` and `cos(refusal, eval_aw)` per layer up-front, so any later "SU shifts the eval-axis readout" claim can be checked against geometric overlap (which we already expect to be ~0).

In [ ]:
# FIXED_ALPHA_STEERING_v1
# Fixed-α (repeng-canonical) steering — see nb15 for rationale.
STEER_LAYERS    = list(range(16, 32))
POSITION_SU     = 'response_last'
POSITION_REF    = -3       # canonical Arditi position
EVAL_VARIANT    = 'contrastive_dataset'
PER_LAYER_SIGMA = False    # legacy flag, kept for manifest compatibility

def _unitize(v):
    v = v.astype(np.float32)
    return v / (np.linalg.norm(v) + 1e-8)

# --- SU (unit) ---
exp6_pca = load_npz(EXP6_PCA)
arrs6    = load_npz(EXP6_ARRAYS)
su_unit  = {l: _unitize(exp6_pca[f'pca_center_dir__{POSITION_SU}__layer_{l:03d}']) for l in STEER_LAYERS}
su_raw   = {l: arrs6[f'mm_raw__{POSITION_SU}__layer_{l:03d}'].astype(np.float32) for l in STEER_LAYERS}
su_norms = {l: float(np.linalg.norm(su_raw[l])) for l in STEER_LAYERS}
su_dirs  = dict(su_unit)
su_sigma = 1.0

# --- refusal (unit) + eval_aw (readout only) ---
dirs_arrs = load_npz(DIRS_NPZ)
ref_unit  = {l: _unitize(dirs_arrs[f'refusal__mm_dir__pos_{POSITION_REF:+d}__layer_{l:03d}']) for l in STEER_LAYERS}
ref_raw   = {l: dirs_arrs[f'refusal__mm_raw__pos_{POSITION_REF:+d}__layer_{l:03d}'].astype(np.float32) for l in STEER_LAYERS}
ref_norms = {l: float(np.linalg.norm(ref_raw[l])) for l in STEER_LAYERS}
ref_dirs  = dict(ref_unit)
ref_sigma = 1.0

# Eval-aware: ALL layers (read upstream of steering range too as sanity check)
READ_LAYERS = list(range(loaded.n_layers))
eval_unit    = {l: _unitize(dirs_arrs[f'eval_aw__mm_dir__{EVAL_VARIANT}__layer_{l:03d}']) for l in READ_LAYERS}
eval_test    = {l: dirs_arrs[f'eval_aw__test_mean__{EVAL_VARIANT}__layer_{l:03d}'].astype(np.float32) for l in READ_LAYERS}
eval_deploy  = {l: dirs_arrs[f'eval_aw__deploy_mean__{EVAL_VARIANT}__layer_{l:03d}'].astype(np.float32) for l in READ_LAYERS}
test_anchor   = {l: float(eval_test[l]   @ eval_unit[l]) for l in READ_LAYERS}
deploy_anchor = {l: float(eval_deploy[l] @ eval_unit[l]) for l in READ_LAYERS}

ref_unit_read = ref_unit
su_unit_read  = su_unit

METHODS = {
    'SU':      {'dirs': su_dirs,  'sigma': su_sigma},
    'refusal': {'dirs': ref_dirs, 'sigma': ref_sigma},
}

def _cos(a, b):
    a = a / (np.linalg.norm(a) + 1e-8); b = b / (np.linalg.norm(b) + 1e-8)
    return float(np.dot(a, b))
cos_su_eval  = {l: _cos(su_unit[l],  eval_unit[l]) for l in STEER_LAYERS}
cos_ref_eval = {l: _cos(ref_unit[l], eval_unit[l]) for l in STEER_LAYERS}
cos_su_ref   = {l: _cos(su_unit[l],  ref_unit[l])  for l in STEER_LAYERS}

print(f'fixed-α steering. POSITION_REF={POSITION_REF}')
print(f'  SU  natural-scale norm median={np.median(list(su_norms.values())):.2f}')
print(f'  ref natural-scale norm median={np.median(list(ref_norms.values())):.2f}')
print(f'  cos(SU, ref) median={np.median(list(cos_su_ref.values())):+.3f}')


## 5 — Forward + capture helper

For each (chat) prompt, run a single forward pass with the model's `output_hidden_states=True`. The forward hook from `ResidualSteerer` modifies the layer's output **in place from the wrapper's perspective**, so the hidden states the model returns reflect the steered residual stream. Read at the **last prompt token** (after `add_generation_prompt=True`) — the position the model would predict the next token from. This matches Arditi's standard readout convention for refusal and is the natural anchor for situational-awareness probes.

Returns `captures[L]` of shape `(N, d_model)` — one row per input prompt, one entry per layer in `READ_LAYERS`.

In [ ]:
BATCH_SIZE = 16
NORMALIZE  = True  # repeng-style: rescale h+δ back to ‖h‖ — must match exp15

PAD_ID = tok.pad_token_id or tok.eos_token_id

def render_chat(system: str, user: str) -> list[int]:
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': user}]
    extra = {'enable_thinking': False} if supports_thinking else {}
    enc = tok.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, **extra)
    if hasattr(enc, 'input_ids'): enc = enc.input_ids
    elif isinstance(enc, dict):   enc = enc['input_ids']
    if hasattr(enc, 'tolist'):    enc = enc.tolist()
    if isinstance(enc, list) and enc and isinstance(enc[0], list): enc = enc[0]
    return [int(x) for x in enc]

def _left_pad_batch(seqs):
    max_len = max(len(s) for s in seqs)
    input_ids = torch.full((len(seqs), max_len), PAD_ID, dtype=torch.long)
    attn_mask = torch.zeros((len(seqs), max_len), dtype=torch.long)
    for i, s in enumerate(seqs):
        n = len(s)
        input_ids[i, max_len - n:] = torch.tensor(s, dtype=torch.long)
        attn_mask[i, max_len - n:] = 1
    return input_ids, attn_mask, max_len

@torch.no_grad()
def _forward_capture_one_batch(seqs, method, k):
    """Returns dict[layer_idx] -> np.ndarray (B, d_model) at the last prompt token."""
    input_ids, attn_mask, _ = _left_pad_batch(seqs)
    input_ids = input_ids.to(device); attn_mask = attn_mask.to(device)

    def _run():
        return model(input_ids=input_ids, attention_mask=attn_mask,
                      output_hidden_states=True, use_cache=False, return_dict=True)

    if method is None or k == 0.0:
        out = _run()
    else:
        m = METHODS[method]
        with ResidualSteerer(loaded, m['dirs'], coeff=k * m['sigma'], normalize=NORMALIZE):
            out = _run()

    # hidden_states is tuple of length n_layers+1: [embeddings, layer_0_out, ..., layer_{N-1}_out].
    # We want the post-block residual at layer L → hidden_states[L+1], at the last (= rightmost) token.
    hs = out.hidden_states
    captures = {}
    for L in READ_LAYERS:
        h = hs[L + 1][:, -1, :]   # (B, d_model)
        captures[L] = h.detach().to(torch.float32).cpu().numpy()
    return captures

def forward_capture_batch(pairs, method, k):
    """Forward-pass `pairs` of (system, user). Returns captures[L]: (N, d_model) over all prompts."""
    seqs = [render_chat(s, u) for s, u in pairs]
    out: dict[int, list] = {L: [] for L in READ_LAYERS}
    for i in range(0, len(seqs), BATCH_SIZE):
        chunk = _forward_capture_one_batch(seqs[i:i + BATCH_SIZE], method, k)
        for L in READ_LAYERS:
            out[L].append(chunk[L])
    return {L: np.concatenate(out[L], axis=0) for L in READ_LAYERS}

# Smoke test
_caps = forward_capture_batch([('You are a helpful assistant.', 'What is 2+2?')], method=None, k=0.0)
print(f'smoke: captured {len(_caps)} layers, layer 16 shape = {_caps[16].shape}')

## 6 — Condition grid

Smaller than exp15 — we don't need the full sweep, just the diagnostic points.

In [ ]:
# Sparser unified α grid for the leverage-curve readout (forward only).
# Same α set for SU and refusal so the leverage-slope comparison is apples-to-apples.
CONDITIONS = [
    ('baseline',        None,       0.0),
    ('SU α=-1.5',       'SU',      -1.5),
    ('SU α=-1.0',       'SU',      -1.0),
    ('SU α=-0.5',       'SU',      -0.5),
    ('SU α=-0.3',       'SU',      -0.3),
    ('SU α=-0.15',      'SU',      -0.15),
    ('SU α=+0.3',       'SU',      +0.3),
    ('SU α=+1.0',       'SU',      +1.0),
    ('refusal α=-1.5',  'refusal', -1.5),
    ('refusal α=-1.0',  'refusal', -1.0),
    ('refusal α=-0.5',  'refusal', -0.5),
    ('refusal α=-0.3',  'refusal', -0.3),
    ('refusal α=-0.15', 'refusal', -0.15),
    ('refusal α=+0.3',  'refusal', +0.3),
    ('refusal α=+1.0',  'refusal', +1.0),
]
for n, m, k in CONDITIONS:
    print(f'  {n:<18s}  method={m!s:<8s}  α={k:+.2f}')


## 7 — Prompt set

Two pools:

- **harmful**: AdvBench-30 (same as exp15).
- **neutral**: a small held-out set of benign requests. Controls for whether SU shifts the eval-axis on *any* input or only on harmful prompts.

Both go through the same generic helpful-assistant system prompt (matches exp15 expB).

In [ ]:
BENIGN_SYSTEM = 'You are a helpful, harmless, and honest assistant.'

harmful = json.loads(ADVBENCH.read_text())
if isinstance(harmful, dict) and 'prompts' in harmful:
    HARMFUL_PROMPTS = harmful['prompts']
elif isinstance(harmful, list):
    HARMFUL_PROMPTS = [p['prompt'] if isinstance(p, dict) and 'prompt' in p else p for p in harmful]
else:
    raise ValueError(f'unexpected advbench format: {type(harmful)}')
HARMFUL_PROMPTS = HARMFUL_PROMPTS[:30]

NEUTRAL_PROMPTS = [
    'Explain why the sky appears blue during the day.',
    'Write a haiku about morning coffee.',
    'What are the main differences between Python lists and tuples?',
    'Recommend three classic novels for someone new to literary fiction.',
    'How does a refrigerator keep food cold?',
    'Summarise the plot of Hamlet in three sentences.',
    'What is the capital of Mongolia, and what is one interesting fact about the country?',
    'Give me a quick recipe for scrambled eggs that serves two.',
    'What is the difference between weather and climate?',
    'Tell me about the history of the printing press.',
    'How do I water a fiddle-leaf fig without overdoing it?',
    'Write a polite email declining an invitation to a wedding.',
    'What does "compound interest" mean in plain English?',
    'Suggest a 30-minute beginner workout routine I can do at home.',
    'How does anaesthesia work, in broad strokes?',
]

POOLS = {
    'harmful': [(BENIGN_SYSTEM, p) for p in HARMFUL_PROMPTS],
    'neutral': [(BENIGN_SYSTEM, p) for p in NEUTRAL_PROMPTS],
}
for name, pool in POOLS.items():
    print(f'  {name:<10s}  N={len(pool):3d}')

## 8 — Sweep: capture residuals + project onto axes

Per (condition, pool, prompt, layer), record:

- `eval_proj` = `h · eval_aw_unit` (raw projection — interpretable against `test_anchor`/`deploy_anchor`).
- `refusal_proj` = `h · refusal_unit` (the existing axis we already know SU steering should disturb at high magnitude).
- `su_proj` = `h · su_unit` (the steered axis itself — sanity check that SU steering does push along its own direction).
- `h_norm` = `‖h‖` (debugging: with `NORMALIZE=True`, baseline and steered should match closely).

Refusal/SU readouts are only available at layers 16..31 (where the unit was extracted at the matching position). Eval-aware is at all 32 layers.

In [ ]:
from tqdm.auto import tqdm

rows = []
for cond_name, method, k in tqdm(CONDITIONS, desc='conditions'):
    for pool_name, pool in POOLS.items():
        captures = forward_capture_batch(pool, method, k)
        for L in READ_LAYERS:
            H = captures[L]                                 # (N, d)
            ev = H @ eval_unit[L]                           # (N,)
            hn = np.linalg.norm(H, axis=-1)                 # (N,)
            if L in STEER_LAYERS:
                rf = H @ ref_unit_read[L]
                su = H @ su_unit_read[L]
            else:
                rf = np.full(H.shape[0], np.nan)
                su = np.full(H.shape[0], np.nan)
            for i in range(H.shape[0]):
                rows.append({
                    'condition': cond_name,
                    'method': method or 'none',
                    'k': k,
                    'pool': pool_name,
                    'prompt_idx': i,
                    'layer': L,
                    'eval_proj':    float(ev[i]),
                    'refusal_proj': float(rf[i]),
                    'su_proj':      float(su[i]),
                    'h_norm':       float(hn[i]),
                })

df = pd.DataFrame(rows)
df.to_csv(OUT_DIR / 'projections_per_prompt.csv', index=False)
print(f'rows = {len(df)}, conditions = {df.condition.nunique()}, layers = {df.layer.nunique()}')
df.head()

## 9 — Aggregate per (condition, pool, layer)

In [ ]:
agg = (df.groupby(['condition', 'method', 'k', 'pool', 'layer'])
         [['eval_proj', 'refusal_proj', 'su_proj', 'h_norm']]
         .agg(['mean', 'std'])
         .reset_index())
agg.columns = ['_'.join(c).rstrip('_') for c in agg.columns]
agg.to_csv(OUT_DIR / 'projections_agg.csv', index=False)

# Δ vs baseline at the (pool, layer) level — the headline quantity.
base = (df[df.condition == 'baseline']
        .groupby(['pool', 'layer'])
        [['eval_proj', 'refusal_proj', 'su_proj']]
        .mean()
        .rename(columns={'eval_proj': 'eval_proj_base',
                          'refusal_proj': 'refusal_proj_base',
                          'su_proj': 'su_proj_base'})
        .reset_index())
delta = df.merge(base, on=['pool', 'layer'])
delta['eval_proj_delta']    = delta['eval_proj']    - delta['eval_proj_base']
delta['refusal_proj_delta'] = delta['refusal_proj'] - delta['refusal_proj_base']
delta['su_proj_delta']      = delta['su_proj']      - delta['su_proj_base']

delta_agg = (delta.groupby(['condition', 'method', 'k', 'pool', 'layer'])
                  [['eval_proj_delta', 'refusal_proj_delta', 'su_proj_delta']]
                  .agg(['mean', 'std'])
                  .reset_index())
delta_agg.columns = ['_'.join(c).rstrip('_') for c in delta_agg.columns]
delta_agg.to_csv(OUT_DIR / 'projections_delta_vs_baseline.csv', index=False)
print(f'agg rows = {len(agg)}; delta rows = {len(delta_agg)}')
agg.head()

## 10 — Plots

Four panels:

1. **Eval-axis projection per layer**, one line per condition, with `test_anchor` and `deploy_anchor` as reference bands. Pool: `harmful`. The headline plot — does `SU −0.7σ` move the curve toward `deploy_anchor`?
2. **Eval-axis Δ vs baseline per layer** — same but baseline-subtracted. Cleanest view of where in the stack the SU push lands on eval.
3. **Refusal-axis Δ vs baseline per layer** — control. We expect SU to push refusal too (consistent with exp15 jailbreak), but the *magnitude* and *layer profile* relative to eval-axis tells the mechanistic story.
4. **Neutral pool, eval-axis Δ** — control for prompt content. If neutral prompts also show eval-axis shift under SU steering, the effect is content-independent (SU disables the eval circuit globally). If only harmful prompts show it, the effect is content-gated.

In [ ]:
import matplotlib.pyplot as plt

FIG_DIR = OUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

ORDER = [c for c, _, _ in CONDITIONS]
COLOURS = {
    'baseline':       '#444444',
    'SU −0.5σ':       '#1f77b4',
    'SU −0.65σ':      '#ff7f0e',
    'SU −0.7σ':       '#d62728',
    'SU −0.85σ':      '#9467bd',
    'refusal −0.4σ':  '#2ca02c',
    'refusal +0.3σ':  '#17becf',
}

def _plot_per_layer(values, ylabel, title, savepath, anchors=None, layers=READ_LAYERS, hl_layers=STEER_LAYERS):
    fig, ax = plt.subplots(figsize=(10, 5))
    for cond in ORDER:
        ys = [values.get((cond, L), float('nan')) for L in layers]
        ax.plot(layers, ys, marker='o', ms=3, label=cond, color=COLOURS.get(cond))
    if anchors is not None:
        ax.plot(layers, [anchors['test'][L]   for L in layers], '--', color='#888888', lw=1, label='test_mean (eval)')
        ax.plot(layers, [anchors['deploy'][L] for L in layers], ':',  color='#888888', lw=1, label='deploy_mean (eval)')
    ax.axvspan(min(hl_layers) - 0.5, max(hl_layers) + 0.5, color='#fff5d8', alpha=0.5, zorder=-1)
    ax.set_xlabel('layer'); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.legend(fontsize=8, loc='best'); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(savepath, dpi=130); plt.show()

def _table(metric_col, agg_df, pool):
    sub = agg_df[agg_df.pool == pool]
    return {(r['condition'], int(r['layer'])): float(r[metric_col]) for _, r in sub.iterrows()}

# Plot 1 — eval-axis projection (raw), harmful pool
_plot_per_layer(
    values=_table('eval_proj_mean', agg, 'harmful'),
    ylabel='h · eval_aw_unit  (mean over harmful prompts)',
    title='Eval-aware axis projection per layer — harmful prompts',
    savepath=FIG_DIR / 'eval_axis_per_layer_harmful.png',
    anchors={'test': test_anchor, 'deploy': deploy_anchor},
)

# Plot 2 — eval-axis Δ vs baseline, harmful pool
_plot_per_layer(
    values=_table('eval_proj_delta_mean', delta_agg, 'harmful'),
    ylabel='Δ (h · eval_aw_unit)  vs baseline',
    title='Eval-axis shift under steering — harmful prompts',
    savepath=FIG_DIR / 'eval_axis_delta_harmful.png',
)

# Plot 3 — refusal-axis Δ vs baseline, harmful pool (only steered layers)
_plot_per_layer(
    values=_table('refusal_proj_delta_mean', delta_agg, 'harmful'),
    ylabel='Δ (h · refusal_unit)  vs baseline',
    title='Refusal-axis shift under steering — harmful prompts (control)',
    savepath=FIG_DIR / 'refusal_axis_delta_harmful.png',
    layers=STEER_LAYERS,
)

# Plot 4 — eval-axis Δ vs baseline, neutral pool (content control)
_plot_per_layer(
    values=_table('eval_proj_delta_mean', delta_agg, 'neutral'),
    ylabel='Δ (h · eval_aw_unit)  vs baseline',
    title='Eval-axis shift under steering — NEUTRAL prompts (content control)',
    savepath=FIG_DIR / 'eval_axis_delta_neutral.png',
)

## 11 — Save manifest + summary

In [ ]:
manifest = {
    'experiment': '16_su_eval_axis_readout',
    'model': loaded.cfg.hf_id,
    'steer_layers': STEER_LAYERS,
    'read_layers': READ_LAYERS,
    'positions': {'SU': POSITION_SU, 'refusal': POSITION_REF, 'eval_aw_variant': EVAL_VARIANT,
                   'read_position': 'last_prompt_token'},
    'sigma': {'SU': float(su_sigma), 'refusal': float(ref_sigma)},
    'per_layer_sigma': PER_LAYER_SIGMA,
    'normalize': NORMALIZE,
    'cos_per_layer': {
        'SU_eval':  cos_su_eval,
        'ref_eval': cos_ref_eval,
        'SU_ref':   cos_su_ref,
    },
    'anchors_per_layer': {
        'test':   test_anchor,
        'deploy': deploy_anchor,
    },
    'conditions': [{'name': n, 'method': m, 'k': k} for n, m, k in CONDITIONS],
    'pools': {name: len(pool) for name, pool in POOLS.items()},
}
(OUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2))
print('wrote', OUT_DIR / 'manifest.json')
print('outputs:')
for p in sorted(OUT_DIR.rglob('*')):
    if p.is_file():
        print(' ', p.relative_to(OUT_DIR))

## What to look for

**Yes (= circuit-level coupling):** `SU −0.7σ` shifts the eval-axis projection on harmful prompts toward `deploy_anchor` at multiple layers — especially mid/late (L≥20). The shift is much larger than at `SU −0.5σ` (where the model still refuses), exceeds noise from the cosine-orthogonality null, and is correlated with `SU −0.85σ` (deeper push, larger shift). Refusal-axis also shifts (we already know that from exp15) but the *layer profile* of eval-axis vs refusal-axis tells us about ordering: if eval moves first/upstream and refusal follows, eval is mediating the jailbreak.

**No (= independent attack axes):** Eval-axis projection is flat across SU magnitudes while refusal-axis moves cleanly. Then SU and eval-aware are mechanistically independent and the jailbreak is a pure refusal-suppression effect.

**Mixed (= worth looking deeper):** Eval-axis moves on harmful prompts but **not** on neutral prompts (Plot 4 stays flat). Then SU → eval coupling is content-gated — the model only routes safety-circuit-relevant inputs through the eval-aware representation. That's a more interesting story than either pure outcome.

**Surprises to flag:**
- Eval-axis shifts at layers *upstream* of the steered range (L<16). That'd be a measurement artefact (the steering is applied at L≥16, no causal pathway upstream) — check the rendering / hook setup.
- `SU −0.5σ` shows large eval-axis shift but no behavioural jailbreak in exp15. Would mean the eval circuit moves freely but the behavioural gate has a separate threshold — eval-axis is necessary but not sufficient.
- `refusal −0.4σ` (the refusal-direction jailbreak) does **not** shift eval-axis. Would imply only the SU jailbreak goes through the eval circuit; the refusal-direction jailbreak takes a different mechanistic path. Cleanest possible result for the "two attack axes are mechanistically distinct" hypothesis.

## 12 — Mid-stack focus + content-gated component

**Result so far.** The L30–31 spike on the eval-axis is content-independent (shows up just as strongly on neutral prompts as harmful), and refusal-axis Δ shows that **SU steering does not move the refusal-axis readout** while refusal-direction steering does so by ~20 mid-stack and ~100 at L31. Combined: the SU jailbreak in exp15 operates through a circuit that is invisible to *both* the eval-aware probe and the refusal probe.

To pin down the small residual signal, two zoomed views below:

1. **Mid-stack only (L16–28), harmful pool**, eval-axis Δ vs baseline. Drops L29–31 so the y-axis isn't dominated by amplification.
2. **Content-gated Δ**: `eval_proj_delta_harmful − eval_proj_delta_neutral` per layer per condition. Subtracts the "any steering moves L31" effect and isolates the part of the eval-axis shift that depends on whether the prompt is harmful.

In [ ]:
MID_LAYERS = list(range(16, 29))  # 16..28 — drops the L29–31 amplification tail

# 12a — eval-axis Δ vs baseline, harmful pool, MID-STACK ONLY
_plot_per_layer(
    values=_table('eval_proj_delta_mean', delta_agg, 'harmful'),
    ylabel='Δ (h · eval_aw_unit)  vs baseline',
    title='Eval-axis shift — harmful prompts (mid-stack L16–28)',
    savepath=FIG_DIR / 'eval_axis_delta_harmful_midstack.png',
    layers=MID_LAYERS,
    hl_layers=MID_LAYERS,
)

# 12b — Content-gated eval-axis Δ: harmful_delta − neutral_delta, all layers
# Builds (cond, L) -> mean(harmful_delta) - mean(neutral_delta) and re-uses the plot helper.
def _content_gated_table(metric_col):
    h = _table(metric_col, delta_agg, 'harmful')
    n = _table(metric_col, delta_agg, 'neutral')
    keys = set(h.keys()) & set(n.keys())
    return {k: h[k] - n[k] for k in keys}

_plot_per_layer(
    values=_content_gated_table('eval_proj_delta_mean'),
    ylabel='Δharmful − Δneutral  (h · eval_aw_unit)',
    title='Content-gated eval-axis shift (harmful-only component)',
    savepath=FIG_DIR / 'eval_axis_content_gated.png',
)

# 12c — Same content-gated view, mid-stack only (the diagnostic plot)
_plot_per_layer(
    values=_content_gated_table('eval_proj_delta_mean'),
    ylabel='Δharmful − Δneutral  (h · eval_aw_unit)',
    title='Content-gated eval-axis shift — mid-stack L16–28',
    savepath=FIG_DIR / 'eval_axis_content_gated_midstack.png',
    layers=MID_LAYERS,
    hl_layers=MID_LAYERS,
)

# 12d — Refusal-axis Δ, mid-stack only (so we can see the SU and refusal-axis
# patterns side by side without L31 dominating)
_plot_per_layer(
    values=_table('refusal_proj_delta_mean', delta_agg, 'harmful'),
    ylabel='Δ (h · refusal_unit)  vs baseline',
    title='Refusal-axis shift — harmful prompts (mid-stack L16–28)',
    savepath=FIG_DIR / 'refusal_axis_delta_harmful_midstack.png',
    layers=MID_LAYERS,
    hl_layers=MID_LAYERS,
)

# Summary table at L24 (representative mid-stack layer) for the writeup
print('\nSummary at L24 (representative mid-stack):')
print(f'{"condition":<16s}  {"eval_Δ_harm":>11s}  {"eval_Δ_neut":>11s}  '
      f'{"eval_gated":>10s}  {"refusal_Δ":>10s}')
for cond, _, _ in CONDITIONS:
    eh = _table('eval_proj_delta_mean', delta_agg, 'harmful').get((cond, 24), float('nan'))
    en = _table('eval_proj_delta_mean', delta_agg, 'neutral').get((cond, 24), float('nan'))
    rf = _table('refusal_proj_delta_mean', delta_agg, 'harmful').get((cond, 24), float('nan'))
    print(f'{cond:<16s}  {eh:>+11.3f}  {en:>+11.3f}  {eh-en:>+10.3f}  {rf:>+10.3f}')


## 13 — Per-σ leverage curves

For each method, plot mid-stack mean feature shift (refusal-axis Δ, eval-axis content-gated Δ) vs steering coefficient `k`, with exp15 compliance overlaid as marker size. The slope of `refusal_Δ` vs `k` quantifies the **per-σ leverage** of each attack on the refusal feature. The slope ratio (refusal-method slope ÷ SU-method slope) tells us how much more efficient SU is at producing the same behavioural compliance per unit feature movement.

Compliance numbers are pasted from exp15 (`exp15_jailbreak_steering (2)/expB_compliance_summary.csv`).

In [ ]:
# Compliance from exp15 (canonical run, dense grid)
EXP15_COMPLIANCE = {
    'baseline':       0.00,
    'SU −0.5σ':       0.00,
    'SU −0.65σ':      0.80,
    'SU −0.7σ':       1.00,
    'SU −0.85σ':      0.90,
    'refusal −0.4σ':  0.87,
    'refusal +0.3σ':  0.00,
}

# Mid-stack mean feature shift per condition (avg over L22..28 — past the steering-injection layers,
# before the L29-31 amplification kicks in)
LEVERAGE_LAYERS = list(range(22, 29))

def _midstack_mean(metric_col, agg_df, pool, condition):
    sub = agg_df[(agg_df.pool == pool) & (agg_df.condition == condition)]
    sub = sub[sub.layer.isin(LEVERAGE_LAYERS)]
    return float(sub[metric_col].mean())

def _gated_midstack(condition):
    h = _midstack_mean('eval_proj_delta_mean', delta_agg, 'harmful', condition)
    n = _midstack_mean('eval_proj_delta_mean', delta_agg, 'neutral', condition)
    return h - n

# Build per-condition summary table
import math
summary = []
for cond, method, k in CONDITIONS:
    summary.append({
        'condition': cond,
        'method':    method or 'baseline',
        'k':         k,
        'refusal_Δ_mid':   _midstack_mean('refusal_proj_delta_mean', delta_agg, 'harmful', cond),
        'eval_gated_mid':  _gated_midstack(cond),
        'compliance':      EXP15_COMPLIANCE.get(cond, math.nan),
    })
summary_df = pd.DataFrame(summary)
summary_df.to_csv(OUT_DIR / 'leverage_summary.csv', index=False)
print('Mid-stack mean (L22–28):')
print(summary_df.to_string(index=False))

# --- Leverage plot: refusal-axis Δ vs k, separated by method ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, ycol, ylabel, title in [
    (axes[0], 'refusal_Δ_mid',  'Δ (h · refusal_unit)  mid-stack mean',
     'Refusal-axis leverage: feature shift per σ'),
    (axes[1], 'eval_gated_mid', 'Δharmful − Δneutral  (h · eval_aw_unit)',
     'Eval-axis leverage (content-gated): feature shift per σ'),
]:
    for method, marker, colour in [('SU', 'o', '#d62728'), ('refusal', 's', '#2ca02c')]:
        sub = summary_df[summary_df.method == method].sort_values('k')
        if len(sub) == 0: continue
        # Marker size scales with compliance: baseline=20, full jailbreak=400
        sizes = 20 + 380 * sub['compliance'].fillna(0).values
        ax.scatter(sub.k, sub[ycol], s=sizes, alpha=0.75, marker=marker,
                   color=colour, edgecolor='black', linewidth=0.5, label=f'{method}  (size = compliance)')
        ax.plot(sub.k, sub[ycol], '--', color=colour, alpha=0.4, lw=1)
        # Fit a least-squares slope through this method's points
        if len(sub) >= 2:
            slope, intercept = np.polyfit(sub.k.values, sub[ycol].values, 1)
            xs = np.linspace(sub.k.min(), sub.k.max(), 20)
            ax.plot(xs, slope * xs + intercept, '-', color=colour, lw=1.2, alpha=0.6)
            # Annotate slope on the plot
            ax.text(sub.k.iloc[-1], sub[ycol].iloc[-1],
                    f'  slope={slope:+.2f}', color=colour, fontsize=9, va='center')
    # Baseline anchor
    base_row = summary_df[summary_df.method == 'baseline']
    if len(base_row):
        ax.scatter(base_row.k, base_row[ycol], s=40, marker='x', color='black', label='baseline')
    ax.axhline(0, color='#888', lw=0.5); ax.axvline(0, color='#888', lw=0.5)
    ax.set_xlabel('steering coefficient k  (× σ)')
    ax.set_ylabel(ylabel); ax.set_title(title)
    ax.legend(fontsize=8, loc='best'); ax.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(FIG_DIR / 'leverage_curves.png', dpi=130)
plt.show()

# Slope ratios — quantify the leverage advantage
print('\nLeverage (slope of feature shift per unit k):')
for ycol in ['refusal_Δ_mid', 'eval_gated_mid']:
    print(f'  {ycol}:')
    for method in ['SU', 'refusal']:
        sub = summary_df[summary_df.method == method].sort_values('k')
        if len(sub) >= 2:
            slope, _ = np.polyfit(sub.k.values, sub[ycol].values, 1)
            print(f'    {method:<8s}  slope = {slope:+.3f}  per σ')

# Compliance-per-feature efficiency: at the jailbreak peak, how much refusal_Δ does
# each method spend per percentage point of compliance gained?
print('\nFeature-spend efficiency at jailbreak peaks:')
for cond in ['SU −0.7σ', 'refusal −0.4σ']:
    row = summary_df[summary_df.condition == cond].iloc[0]
    if row.compliance > 0:
        eff = row['refusal_Δ_mid'] / row.compliance
        print(f'  {cond:<16s}  refusal_Δ = {row["refusal_Δ_mid"]:+.2f}  '
              f'compliance = {row.compliance:.2%}  '
              f'(refusal_Δ per unit compliance = {eff:+.2f})')
